# Basic RAG foundation

This notebook builds the project in small, inspectable stages: PDF parsing, chunking, embeddings/vector database, LLM generation, and evaluation.

We start with parsing because every later RAG stage must consume validated canonical article JSON, not raw PDF text or model-invented metadata. The parser here follows the `reference.ipynb` approach but calls the reusable repository module so the logic stays testable outside the notebook.

## Reference notebook analysis

`reference.ipynb` implements the full intended pipeline:

1. Defines a strict canonical `Article` contract with article number, Arabic text, optional English text, hierarchy, repeal status, source page, and deterministic citation.
2. Uses `pdfplumber` table geometry instead of naive text extraction because the source PDF is bilingual and column-based.
3. Reconstructs Arabic glyph order line by line, while preserving left-to-right numeric runs.
4. Caches raw extraction rows using the source PDF hash and extractor version.
5. Parses rows into article records, validates the expected article range and repeal ranges, and writes candidates/reports for review.
6. Blocks canonical `articles.json` when validation fails, so chunking and indexing cannot accidentally use ambiguous source data.

For this first step, we only implement the parsing gate. Chunking, vector indexing, LLM generation, and evaluation will be added after the corpus contract is stable.

In [3]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def find_root(start: Path | None = None) -> Path:
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "data/raw").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or notebooks directory.")


ROOT = find_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PDF_PATH = ROOT / "data/raw/egyptian_civil_law.pdf"
PROCESSED = ROOT / "data/processed"

print({"root": str(ROOT), "pdf_exists": PDF_PATH.exists(), "processed": str(PROCESSED)})

{'root': '/Users/abdelhakimosama/Projects/ML_DA_Labs/MLOPs/arabic_legal_doc_QA', 'pdf_exists': True, 'processed': '/Users/abdelhakimosama/Projects/ML_DA_Labs/MLOPs/arabic_legal_doc_QA/data/processed'}


## 1. Parse PDF into JSON

The parser writes three review artifacts every time:

- `data/processed/extraction_rows.json`: cached table cells and diagnostics keyed by PDF hash and extractor version.
- `data/processed/article_candidates.json`: best parsed article candidates for inspection, even when validation fails.
- `data/processed/extraction_report.json`: validation errors, warnings, source hash, diagnostics, and article count.

It writes `data/processed/articles.json` only when the complete validation gate passes. That file is the canonical corpus for chunking and every downstream RAG stage.

In [4]:
from arabic_legal_qa.rag import pdf_loader

print({
    "extractor_version": pdf_loader.EXTRACTOR_VERSION,
    "expected_articles": [min(pdf_loader.EXPECTED_ARTICLES), max(pdf_loader.EXPECTED_ARTICLES)],
    "repeal_ranges": pdf_loader.REPEAL_RANGES,
})

{'extractor_version': 'geometry-glyphs-v1', 'expected_articles': [1, 1149], 'repeal_ranges': ((54, 80), (389, 417))}


In [5]:
articles, extraction_report = pdf_loader.extract_to_json(PDF_PATH, PROCESSED)

summary = {
    "valid": extraction_report["valid"],
    "article_candidates": len(articles),
    "repealed_candidates": sum(article.is_repealed for article in articles),
    "errors": extraction_report["errors"][:8],
    "warnings": extraction_report["warnings"][:8],
}
summary

{'valid': True,
 'article_candidates': 1149,
 'repealed_candidates': 56,
 'errors': [],
 'warnings': ["Page 59: recognized printed 'rticle 452' using the matching Arabic heading",
  'Bilingual heading mismatch at page 147: EN 1022, AR None; handled by explicit source correction',
  'Page 147: applied explicit Arabic source correction for Article 1022']}

### Interpreting the parsing result

A `valid: false` result is not a notebook failure. It means the parser produced reviewable candidates but refused to publish canonical `articles.json`. That is the behavior we want for legal RAG: ambiguous extraction should stop before chunking and indexing.

This source edition has a known Arabic omission around Article 1022. The loader applies an explicit Arabic correction and records the correction in `extraction_report.json`, so downstream cells can consume a complete canonical corpus with provenance.

In [ ]:
def article_preview(article: pdf_loader.Article) -> dict[str, object]:
    return {
        "article_number": article.article_number,
        "citation": article.citation,
        "source_page": article.source_page,
        "is_repealed": article.is_repealed,
        "book": article.book,
        "chapter": article.chapter,
        "text_ar_preview": article.text_ar[:180],
        "text_en_preview": (article.text_en or "")[:180],
    }


for number in (1, 54, 80, 147, 389, 417, 1021, 1022):
    match = next((article for article in articles if article.article_number == number), None)
    if match:
        print(json.dumps(article_preview(match), ensure_ascii=False, indent=2))

In [ ]:
try:
    canonical_articles = pdf_loader.load_validated_articles(PROCESSED, PDF_PATH)
except ValueError as exc:
    canonical_articles = None
    print(f"Canonical corpus is not available yet: {exc}")
else:
    corpus_hash = pdf_loader.stable_hash([article.model_dump() for article in canonical_articles])
    print({"canonical_articles": len(canonical_articles), "corpus_hash": corpus_hash})

## Next implementation step

After the parsing gate is accepted, the next notebook section should add article-aware chunking. Each chunk must carry parent article number, citation, language, source page, hierarchy, repeal status, and corpus/index version metadata.